# 18. 프롬프트 + 거절 규칙 최적화 (v2.1): 학습 없이 마지막 끌어올리기

16(검색·생성 모델)·17(DPO) 중 **검증 정답률이 가장 높은 모델**을 고정하고, 남은 두 가지를 검증셋으로 고릅니다.

| 단계 | 비교 | 노리는 것 |
|---|---|---|
| ① 프롬프트 | **P0** 지금 / **P1** 문단이 일부만 답해도 그 부분은 답하기 / **P2** P1 + 핵심 답 먼저 | 잘못 거절(.056)·부분 정답 줄이기 |
| ② 거절 규칙 | MiniLM 최고점 4.92 (08) vs **MiniLM·bge-reranker-v2-m3 최고점 기준 여러 개** | 진짜 답 없는 질문 거절 ↑, 답 있는 질문 거절 ↓ |

- 거절 규칙을 바꿔 보려고 **모든 질문에 답을 생성** (규칙은 나중에 덮어씀)
- 검증 = 답 있음 200 + **답 없음 100** (14 검증 생성셋). 답 없음은 심판이 "문단에 답 없음(< 0.2)"으로 본 질문만 **진짜 답 없음**으로 셈
- 고르는 기준 (검증): **균형 점수 = (정답률 + 진짜 답 없음 거절률) / 2**, 시험 650은 보고용
- 결과: `final/final_config.json` → 08c 챗봇에 그대로 적용

## 1. 설치 + 경로

In [ ]:
!pip -q install -U peft faiss-cpu sentence-transformers
!pip -q uninstall -y torchao   # 최신 peft가 Colab 기본 torchao(0.10)와 충돌해서 제거 (14·15에서 확인한 조합)

In [ ]:
QUICK = False   # True: 질문 수십 개로 점검 (결과는 final/quick) / False: 본 실행

import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
RAG_DIR    = f'{SAVE_DIR}/rag'          # 08 시험 답변
R2_DIR     = f'{SAVE_DIR}/rerank2'      # 11 후보·재정렬 점수
JUDGE_DIR  = f'{SAVE_DIR}/judge_eval'   # 14 검증 답변 + 심판 캐시
GC_DIR     = f'{SAVE_DIR}/gen_compare'                              # 16 본 실행 결과 (QUICK도 이걸 씀)
DPO_DIR    = f'{SAVE_DIR}/dpo'                                      # 17 본 실행 결과 (QUICK도 이걸 씀)
OUT_DIR    = f'{SAVE_DIR}/final' + ('/quick' if QUICK else '')
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(REPORT_DIR, exist_ok=True)
print('QUICK =', QUICK, '→', OUT_DIR)

## 2. 설정 + 공통 함수 (16과 같음)

In [ ]:
import re, json, gzip, time, random, string, hashlib, gc
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch, transformers, peft

SEED      = 42
THRESH    = 4.921875
ENSEMBLE  = ['bge-reranker-large', 'bge-reranker-v2-m3']
BASE_RR   = 'ms-marco-MiniLM-L-6-v2'
BASE_GEN  = 'Qwen/Qwen2.5-7B-Instruct'
GENS      = ['Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen3-8B', 'Qwen/Qwen3-14B']
JUDGE_LLM = 'Qwen/Qwen2.5-14B-Instruct'
JUDGE2    = 'microsoft/phi-4'
GEN_BS    = 16
N_TEST_A, N_TEST_N, N_TEST_O, N_VAL = (20, 5, 5, 12) if QUICK else (None, None, None, None)
DEMO = ['why does my face turn red and itchy after drinking alcohol',
        'what are the symptoms of low vitamin d',
        'can ibuprofen cause stomach bleeding',
        'how long does the flu last',
        'what is a normal resting heart rate for adults']

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')
print(f'torch {torch.__version__} / transformers {transformers.__version__} / peft {peft.__version__}')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower(); s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref); c = sum((Counter(p) & Counter(r)).values())
    if not p or not r or c == 0:
        return 0.0
    pr, rc = c / len(p), c / len(r)
    return 2 * pr * rc / (pr + rc)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

SYS_RAG = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences."""

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f'GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h
sites  = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())
docs_text = lambda pids: '\n'.join(f'[{i}] ({sites[p]}) {texts[p]}' for i, p in enumerate(pids, 1))
def rag_messages(q, pids):
    return [{'role': 'system', 'content': SYS_RAG}, {'role': 'user', 'content': f'Documents:\n{docs_text(pids)}\n\nQuestion: {q}'}]
print(f'문단 {len(texts):,}')

SYS_P1 = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the documents answer the question only partly, answer with what they do say. Do not refuse in that case.
4. Only if the question is not about health or medicine, or no document contains information that answers it, reply with exactly this sentence: "{REFUSAL}"
5. Answer in 2-4 short sentences."""
SYS_P2 = SYS_P1.replace('5. Answer in 2-4 short sentences.',
                        '5. Start with one sentence that directly answers the question, then add 1-3 short supporting sentences.')
PROMPTS = {'P0': SYS_RAG, 'P1': SYS_P1, 'P2': SYS_P2}
def msgs_p(q, pids, sysp):
    return [{'role': 'system', 'content': sysp}, {'role': 'user', 'content': f'Documents:\n{docs_text(pids)}\n\nQuestion: {q}'}]

## 3. 질문 + 문단 + 거절 점수

In [ ]:
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
C  = np.load(f'{R2_DIR}/candidates.npz')
SC = {n: np.load(f'{R2_DIR}/scores_{n}.npz') for n in ENSEMBLE + [BASE_RR]}
zs = lambda m: (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def pick(order_scores, ids, n):
    docs, seen = [], set()
    for j in np.argsort(-order_scores):
        p = int(ids[j]); key = norm_text(texts[p])
        if key in seen:
            continue
        seen.add(key); docs.append(p)
        if len(docs) == n:
            break
    return docs

groups = {'test':  [q for q in eval_set if q['domain'] == 'health' and q.get('answerable', True) and q.get('gold_pids')],
          'noans': [q for q in eval_set if q['domain'] == 'health' and not q.get('answerable', True)],
          'other': [q for q in eval_set if q['domain'] != 'health'],
          'val':   val_set}
info = {}
for g, qs in groups.items():
    ids = C[f'{g}_ids']
    assert len(ids) == len(qs), f'{g}: 11 후보 수 {len(ids)} ≠ 질문 수 {len(qs)}'
    ens = (zs(SC[ENSEMBLE[0]][g]) + zs(SC[ENSEMBLE[1]][g])) / 2
    base = SC[BASE_RR][g]
    for i, q in enumerate(qs):
        info[(g == 'val', str(q['qid']))] = {'top_minilm': float(base[i].max()), 'M5': pick(base[i], ids[i], 5),
                                              'E5': pick(ens[i], ids[i], 5), 'E10': pick(ens[i], ids[i], 10)}

# 시험 650: 08 답변 (S0 = 'RAG 개선', 원본 7B + MiniLM 상위 5)
evq = {str(q['qid']): q for q in eval_set}
test_rows = []
for r in load_jsonl(f'{RAG_DIR}/eval_answers.jsonl'):
    q = evq[str(r['qid'])]; inf = info[(False, str(r['qid']))]
    g = 'A' if r['domain'] == 'health' and r['answerable'] else ('N' if r['domain'] == 'health' else 'O')
    test_rows.append({'qid': str(r['qid']), 'query': r['query'], 'answers': q['answers'], 'gold_pids': q.get('gold_pids') or [],
                      'group': g, 'refuse': r['rag_docs'][0]['score'] < THRESH,
                      'docs': {'S0': [d['pid'] for d in r['rag_docs']], 'E5': inf['E5'], 'E10': inf['E10']},
                      'ans': {'S0|7B': r['RAG 개선']}})
# 검증 200 (답 있음): 14 답변 (S0 = 'RAG 개선')
v14 = {str(r['qid']): r for r in load_jsonl(f'{JUDGE_DIR}/val_answers.jsonl') if r['group'] == 'A'}
val_rows = []
for q in val_set:
    r = v14.get(str(q['qid'])); inf = info[(True, str(q['qid']))]
    if r is None:
        continue
    val_rows.append({'qid': str(q['qid']), 'query': q['query'], 'answers': r['answers'], 'gold_pids': q.get('gold_pids') or [],
                     'group': 'A', 'refuse': r['rag_docs'][0]['score'] < THRESH,
                     'docs': {'S0': [d['pid'] for d in r['rag_docs']], 'E5': inf['E5'], 'E10': inf['E10']},
                     'ans': {'S0|7B': r['RAG 개선']}})
if QUICK:
    by = lambda g: [r for r in test_rows if r['group'] == g]
    test_rows = by('A')[:N_TEST_A] + by('N')[:N_TEST_N] + by('O')[:N_TEST_O]
    val_rows = val_rows[:N_VAL]
print(f'시험 {len(test_rows)} {Counter(r["group"] for r in test_rows)} / 검증 {len(val_rows)}')
for k in ('S0', 'E5', 'E10'):
    for name, rows in [('시험', test_rows), ('검증', val_rows)]:
        A = [r for r in rows if r['group'] == 'A']
        print(f'  {k} {name} 근거 R@N {np.mean([bool(set(r["gold_pids"]) & set(r["docs"][k])) for r in A]):.3f}', end='')
    print()

ANS_PATH = f'{OUT_DIR}/answers.jsonl'
def save_answers():
    save_jsonl([{'set': s, 'qid': r['qid'], 'ans': r['ans']} for s, rows in [('test', test_rows), ('val', val_rows)] for r in rows], ANS_PATH)
if os.path.exists(ANS_PATH):
    saved = {(d['set'], d['qid']): d['ans'] for d in load_jsonl(ANS_PATH)}
    for s, rows in [('test', test_rows), ('val', val_rows)]:
        for r in rows:
            r['ans'].update(saved.get((s, r['qid']), {}))
    print('저장된 답변 불러옴:', sorted({k for r in test_rows for k in r['ans']}))

# 거절 점수: MiniLM·v2-m3 최고점 (11 점수)
for g, qs in groups.items():
    mini, m3 = SC[BASE_RR][g], SC['bge-reranker-v2-m3'][g]
    for i, q in enumerate(qs):
        info[(g == 'val', str(q['qid']))].update({'s_mini': float(mini[i].max()), 's_m3': float(m3[i].max())})
for rows, is_val in ((test_rows, False), (val_rows, True)):
    for r in rows:
        inf = info[(is_val, r['qid'])]
        r['s_mini'], r['s_m3'] = inf['s_mini'], inf['s_m3']

# 검증 답 없음 100: 11에 없어서 여기서 검색 + 재정렬
N_VALN = 4 if QUICK else 100
vN = [r for r in load_jsonl(f'{JUDGE_DIR}/val_answers.jsonl') if r['group'] == 'N'][:N_VALN]
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
index = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
emb = SentenceTransformer('BAAI/bge-base-en-v1.5', device='cuda').half()
qs_ = [r['query'] for r in vN]
qe = emb.encode(['Represent this sentence for searching relevant passages: ' + q for q in qs_],
                normalize_embeddings=True, convert_to_numpy=True).astype('float32')
_, ids = index.search(qe, 100)
pairs = [(q, texts[p]) for q, c in zip(qs_, ids.tolist()) for p in c]
def ce_load(name, max_len):
    # 11과 같이 원래 점수(활성화 함수 없음)로: 거절 기준 단위를 맞추기 위해
    try:
        return CrossEncoder(name, max_length=max_len, device='cuda', activation_fn=torch.nn.Identity())
    except TypeError:
        return CrossEncoder(name, max_length=max_len, device='cuda', default_activation_function=torch.nn.Identity())
mini = ce_load('cross-encoder/ms-marco-MiniLM-L-6-v2', 256).predict(pairs, batch_size=256).reshape(len(qs_), 100)
raw = {}
for m in ['BAAI/bge-reranker-large', 'BAAI/bge-reranker-v2-m3']:
    ce = ce_load(m, 512); ce.model.half()
    raw[m] = ce.predict(pairs, batch_size=64).reshape(len(qs_), 100); del ce
ens = (zs(raw['BAAI/bge-reranker-large']) + zs(raw['BAAI/bge-reranker-v2-m3'])) / 2
del emb, index; free_gpu()
for i, r in enumerate(vN):
    val_rows.append({'qid': str(r['qid']), 'query': r['query'], 'answers': [], 'gold_pids': [], 'group': 'N',
                     'refuse': mini[i].max() < THRESH, 's_mini': float(mini[i].max()), 's_m3': float(raw['BAAI/bge-reranker-v2-m3'][i].max()),
                     'docs': {'S0': pick(mini[i], ids[i], 5), 'E5': pick(ens[i], ids[i], 5), 'E10': pick(ens[i], ids[i], 10)},
                     'ans': {}})
print(f'검증 {len(val_rows)} {Counter(r["group"] for r in val_rows)}')
print('검증 답 없음 MiniLM 최고점 범위 (11 단위와 같아야 함, 대략 -3~10):', round(float(mini.max(1).min()), 2), '~', round(float(mini.max(1).max()), 2))

## 4. 생성·심판 함수 (16과 같음)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_gen(name):
    tok = AutoTokenizer.from_pretrained(name, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
    return tok, llm

@torch.inference_mode()
def generate(tok, llm, msgs_list, bs=GEN_BS, max_new=256):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                   for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
        dec = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        outs += [re.sub(r'<think>.*?</think>', '', o, flags=re.S).strip() for o in dec]
    return outs

def run_system(tok, llm, key, docs_key):
    # key = '검색|모델' 시스템 답변을 시험·검증 모두 생성 (거절 규칙 08과 같음)
    for rows in (test_rows, val_rows):
        todo = [r for r in rows if not r['refuse'] and key not in r['ans']]
        t = time.time()
        outs = generate(tok, llm, [rag_messages(r['query'], r['docs'][docs_key]) for r in todo])
        for r, a in zip(todo, outs):
            r['ans'][key] = a
        for r in rows:
            if r['refuse']:
                r['ans'][key] = REFUSAL
        if todo:
            print(f'  {key}: {len(todo)}개 생성 {time.time() - t:.0f}초')
    save_answers()

SYS_CORRECT = ('You grade answers to health questions. Compare the candidate answer with the reference answer(s). '
               'Reply with one digit only.\n'
               '2 = correct: states the key facts of a reference answer (different wording or extra correct detail is fine)\n'
               '1 = partially correct: some key facts, but incomplete or partly wrong\n'
               '0 = wrong, unrelated, or does not answer the question')
SYS_FAITH = ('You check whether an answer is supported by the given documents. Reply with one digit only.\n'
             '1 = every factual statement in the answer is stated in the documents\n'
             '0 = the answer contains a fact that is not in the documents, or contradicts them')
SYS_EVID = ('You check whether the given documents contain the information needed to answer a health question. '
            'Reply with one digit only.\n'
            '1 = at least one document directly answers the question\n'
            '0 = the documents do not answer the question (related topic only, or missing information)')
def correct_msgs(q, refs, ans):
    ref = '\n'.join(f'- {x}' for x in refs[:3])
    return [{'role': 'system', 'content': SYS_CORRECT},
            {'role': 'user', 'content': f'Question: {q}\nReference answer(s):\n{ref}\nCandidate answer: {ans}\n\nGrade (0, 1 or 2):'}]
def faith_msgs(dt, ans):
    return [{'role': 'system', 'content': SYS_FAITH},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nAnswer: {ans}\n\nSupported (0 or 1)?'}]
def evid_msgs(q, dt):
    return [{'role': 'system', 'content': SYS_EVID},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nQuestion: {q}\n\nDo the documents answer it (0 or 1)?'}]

class Judge:
    def __init__(self, name, cache_path, tag=''):
        self.tok = AutoTokenizer.from_pretrained(name, padding_side='left')
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
        self.digits = [self.tok.encode(d, add_special_tokens=False)[-1] for d in '012']
        self.path, self.tag = cache_path, tag
        self.cache = {j['key']: j for j in load_jsonl(cache_path)} if os.path.exists(cache_path) else {}
    def key(self, kind, q, a):
        return hashlib.md5(f'{self.tag}{kind}|{q}|{a}'.encode()).hexdigest()
    @torch.inference_mode()
    def probs(self, msgs_list, n, bs=8):
        prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
        order = np.argsort([len(p) for p in prompts]); out = [None] * len(prompts)
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            enc = self.tok([prompts[i] for i in idx], return_tensors='pt', padding=True, add_special_tokens=False).to(self.llm.device)
            logits = self.llm(**enc).logits[:, -1, self.digits[:n]].float()
            for i, p in zip(idx, torch.softmax(logits, -1).cpu().numpy()):
                out[i] = p
        return out
    def score(self, kind, items):
        # 'c' (q, refs, ans) → 0~2 / 'f' (q, docs_text, ans) → 0~1 / 'e' (q, docs_text) → 0~1
        keyed = [(self.key(kind, it[0], it[-1]), it) for it in items]
        need = list({k: it for k, it in keyed if k not in self.cache}.items())
        t = time.time()
        for s in range(0, len(need), 2000):
            part = need[s:s + 2000]
            if kind == 'c':
                vals = [float(p[1] + 2 * p[2]) for p in self.probs([correct_msgs(*it) for _, it in part], 3)]
            elif kind == 'f':
                vals = [float(p[1]) for p in self.probs([faith_msgs(it[1], it[2]) for _, it in part], 2)]
            else:
                vals = [float(p[1]) for p in self.probs([evid_msgs(it[0], it[1]) for _, it in part], 2)]
            for (k, _), v in zip(part, vals):
                self.cache[k] = {'key': k, 'v': v}
            save_jsonl(list(self.cache.values()), self.path)
        if need:
            print(f'  심판{self.tag} {kind}: {len(need)}개 새로 채점 ({time.time() - t:.0f}초)')
        return [self.cache[k]['v'] for k, _ in keyed]
    def close(self):
        del self.llm; free_gpu()

def evaluate(J, rows, systems, faith=True):
    table = {}
    for key in systems:
        dk = key.split('|')[0]
        A = [r for r in rows if r['group'] == 'A']
        ansA = [r for r in A if not is_refusal(r['ans'][key])]
        cs = dict(zip([id(r) for r in ansA], J.score('c', [(r['query'], r['answers'], r['ans'][key]) for r in ansA])))
        c_all = [cs.get(id(r), 0.0) for r in A]
        g = lambda k: [r for r in rows if r['group'] == k]
        row = {'정답률 (심판 2점)': np.mean([c >= 1.5 for c in c_all]),
               '정답+부분': np.mean([c >= 0.5 for c in c_all]),
               '평균 점수 (0~2)': np.mean(c_all),
               'F1 (참고)': np.mean([max(f1(r['ans'][key], x) for x in r['answers']) for r in A]),
               '잘못 거절': np.mean([is_refusal(r['ans'][key]) for r in A])}
        if faith:
            answered = [r for r in rows if not is_refusal(r['ans'][key])]
            fs = J.score('f', [(r['query'], docs_text(r['docs'][dk]), r['ans'][key]) for r in answered])
            row['근거 충실도'] = np.mean([f >= 0.5 for f in fs]) if fs else np.nan
        if g('N'):
            row['답 없음 거절률 (MS MARCO 기준)'] = np.mean([is_refusal(r['ans'][key]) for r in g('N')])
        if g('O'):
            row['분야 밖 거절률'] = np.mean([is_refusal(r['ans'][key]) for r in g('O')])
        table[key] = row
    return pd.DataFrame(table).T.round(3)

JUDGE_PATH = f'{JUDGE_DIR}/judgments.jsonl'

## 5. 최종 모델 고르기: 16 최고 vs 17 DPO (검증 정답률, 같은 심판 캐시)

In [ ]:
cfg = json.load(open(f'{GC_DIR}/best_config.json'))
BEST = cfg['best']; RETR = BEST.split('|')[0]
MODEL_OF = {'7B': BASE_GEN, 'Qwen2.5-14B': 'Qwen/Qwen2.5-14B-Instruct', 'Qwen3-8B': 'Qwen/Qwen3-8B', 'Qwen3-14B': 'Qwen/Qwen3-14B'}
GEN_NAME, ADAPTER = MODEL_OF[BEST.split('|')[1]], None
prev = {}
for path in (f'{GC_DIR}/answers.jsonl', f'{DPO_DIR}/answers.jsonl'):
    if os.path.exists(path):
        for d in load_jsonl(path):
            if d['set'] == 'val':
                prev.setdefault(d['qid'], {}).update(d['ans'])
DPO_KEY = f'{BEST}+DPO'
if os.path.exists(f'{DPO_DIR}/adapter/adapter_config.json') and all(DPO_KEY in prev.get(r['qid'], {}) for r in val_rows if r['group'] == 'A'):
    J = Judge(JUDGE_LLM, JUDGE_PATH)
    A = [r for r in val_rows if r['group'] == 'A']
    accs = {}
    for k in (BEST, DPO_KEY):
        cs = J.score('c', [(r['query'], r['answers'], prev[r['qid']][k]) for r in A])
        accs[k] = np.mean([c >= 1.5 and not is_refusal(prev[r['qid']][k]) for r, c in zip(A, cs)])
    J.close(); del J
    print('검증 정답률:', {k: round(v, 3) for k, v in accs.items()})
    if accs[DPO_KEY] > accs[BEST]:
        ADAPTER = f'{DPO_DIR}/adapter'
FINAL = DPO_KEY if ADAPTER else BEST
print(f'최종 모델: {FINAL} (생성 {GEN_NAME}{" + DPO LoRA" if ADAPTER else ""}, 검색 {RETR})')

## 6. 모든 질문에 답 생성 (프롬프트 3종, 거절 규칙 무시)

In [ ]:
from peft import PeftModel
ANS_PATH = f'{OUT_DIR}/answers.jsonl'
if os.path.exists(ANS_PATH):
    saved = {(d['set'], d['qid']): d['ans'] for d in load_jsonl(ANS_PATH)}
    for s, rows in (('test', test_rows), ('val', val_rows)):
        for r in rows:
            r['ans'].update(saved.get((s, r['qid']), {}))
need = [(p, rows) for p in PROMPTS for rows in (test_rows, val_rows) if any(p not in r['ans'] for r in rows)]
if need:
    tok, llm = load_gen(GEN_NAME)
    if ADAPTER:
        llm = PeftModel.from_pretrained(llm, ADAPTER).eval()
    for p, rows in need:
        todo = [r for r in rows if p not in r['ans']]
        t = time.time()
        outs = generate(tok, llm, [msgs_p(r['query'], r['docs'][RETR], PROMPTS[p]) for r in todo])
        for r, a in zip(todo, outs):
            r['ans'][p] = a
        save_answers()
        print(f'  {p}: {len(todo)}개 {time.time() - t:.0f}초')
    del llm; free_gpu()

## 7. 심판 + 프롬프트·거절 규칙 고르기 (검증)

In [ ]:
J = Judge(JUDGE_LLM, JUDGE_PATH)
for rows in (test_rows, val_rows):
    for r in rows:
        r['dt'] = docs_text(r['docs'][RETR])
    A = [r for r in rows if r['group'] == 'A']
    for p in PROMPTS:
        ok = [r for r in A if not is_refusal(r['ans'][p])]
        for r, c in zip(ok, J.score('c', [(r['query'], r['answers'], r['ans'][p]) for r in ok])):
            r.setdefault('c', {})[p] = c
        ok = [r for r in rows if not is_refusal(r['ans'][p])]
        for r, f in zip(ok, J.score('f', [(r['query'], r['dt'], r['ans'][p]) for r in ok])):
            r.setdefault('f', {})[p] = f
    N = [r for r in rows if r['group'] == 'N']
    for r, e in zip(N, J.score('e', [(r['query'], r['dt']) for r in N])):
        r['evid'] = e

def table_for(rows, p, rule):
    # rule(r) == True → 거절 (LLM 안 부름)
    A = [r for r in rows if r['group'] == 'A']; N = [r for r in rows if r['group'] == 'N']; O = [r for r in rows if r['group'] == 'O']
    ans = lambda r: REFUSAL if rule(r) else r['ans'][p]
    cor = [0.0 if is_refusal(ans(r)) else r.get('c', {}).get(p, 0.0) for r in A]
    true_no = [r for r in N if r['evid'] < 0.2]; has = [r for r in N if r['evid'] >= 0.8]
    answered = [r for r in rows if not is_refusal(ans(r))]
    out = {'정답률': np.mean([c >= 1.5 for c in cor]), '정답+부분': np.mean([c >= 0.5 for c in cor]),
           '잘못 거절': np.mean([is_refusal(ans(r)) for r in A]),
           '진짜 답 없음 거절률': np.mean([is_refusal(ans(r)) for r in true_no]) if true_no else np.nan,
           '사실 답 있음 답변률': np.mean([not is_refusal(ans(r)) for r in has]) if has else np.nan,
           '근거 충실도': np.mean([r.get('f', {}).get(p, 0.0) >= 0.5 for r in answered]) if answered else np.nan}
    if O:
        out['분야 밖 거절률'] = np.mean([is_refusal(ans(r)) for r in O])
    out['균형 점수'] = (out['정답률'] + (out['진짜 답 없음 거절률'] if true_no else 0)) / (2 if true_no else 1)
    return out

rules = {'08 규칙 (MiniLM < 4.92)': lambda r: r['s_mini'] < THRESH, '거절 규칙 없음 (LLM이 판단)': lambda r: False}
for sig, col in (('MiniLM', 's_mini'), ('v2-m3', 's_m3')):
    vals = np.array([r[col] for r in val_rows])
    for q in (0.05, 0.10, 0.15, 0.20, 0.25, 0.30):
        th = float(np.quantile(vals, q))
        rules[f'{sig} < {th:.2f} (검증 하위 {int(q * 100)}%)'] = (lambda c, t: (lambda r: r[c] < t))(col, th)

grid = {(p, name): table_for(val_rows, p, rule) for p in PROMPTS for name, rule in rules.items()}
grid_tab = pd.DataFrame(grid).T.round(3)
print('[검증: 프롬프트 × 거절 규칙 (균형 점수 순 상위 12)]')
display(grid_tab.sort_values('균형 점수', ascending=False).head(12))
best_p, best_rule = grid_tab['균형 점수'].idxmax()
print('→ 고른 조합:', best_p, '/', best_rule)

## 8. 시험 650에 적용 (보고용)

In [ ]:
base_key = ('P0', '08 규칙 (MiniLM < 4.92)')
show = list(dict.fromkeys([base_key, ('P1', base_key[1]), ('P2', base_key[1]), (best_p, base_key[1]), (best_p, best_rule)]))
test_tab = pd.DataFrame({f'{p} / {n}': table_for(test_rows, p, rules[n]) for p, n in show}).T.round(3)
val_tab = pd.DataFrame({f'{p} / {n}': table_for(val_rows, p, rules[n]) for p, n in show}).T.round(3)
print('[검증]'); display(val_tab)
print('[시험 650]'); display(test_tab)

# 08 (S0|7B, P0, MiniLM 규칙) 대비 정답률 차이 부트스트랩
A = [r for r in test_rows if r['group'] == 'A']
def ok_final(r):
    a = REFUSAL if rules[best_rule](r) else r['ans'][best_p]
    return (not is_refusal(a)) and r.get('c', {}).get(best_p, 0) >= 1.5
def ok_08(r):
    a = r['ans']['S0|7B']
    return (not is_refusal(a)) and J.cache[J.key('c', r['query'], a)]['v'] >= 1.5
J.score('c', [(r['query'], r['answers'], r['ans']['S0|7B']) for r in A if not is_refusal(r['ans']['S0|7B'])])
d = np.array([ok_final(r) - ok_08(r) for r in A], dtype=float)
bs = np.random.default_rng(SEED).choice(d, (2000, len(d))).mean(1)
ci = (d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5))
print(f'시험 정답률: 최종 - 08 = {ci[0]:+.3f} (95% CI {ci[1]:+.3f} ~ {ci[2]:+.3f})')
J.close(); del J

## 9. 저장 (`final_config.json` → 챗봇에 적용)

In [ ]:
th_rule = best_rule
final_cfg = {'generator': GEN_NAME, 'adapter': ADAPTER.replace('/content/drive/MyDrive/', '') if ADAPTER else None,
             'retrieval': RETR, 'top_n': 10 if RETR == 'E10' else 5,
             'rerankers': ['BAAI/bge-reranker-large', 'BAAI/bge-reranker-v2-m3'] if RETR != 'S0' else ['cross-encoder/ms-marco-MiniLM-L-6-v2'],
             'prompt': best_p, 'system_prompt': PROMPTS[best_p], 'refusal_rule': best_rule,
             'refusal_signal': None if '없음' in best_rule else ('cross-encoder/ms-marco-MiniLM-L-6-v2' if best_rule.startswith(('MiniLM', '08')) else 'BAAI/bge-reranker-v2-m3'),
             'refusal_threshold': None if '없음' in best_rule else (THRESH if best_rule.startswith('08') else float(re.search(r'< (-?[\d.]+)', best_rule).group(1)))}
json.dump(final_cfg, open(f'{OUT_DIR}/final_config.json', 'w'), ensure_ascii=False, indent=1)
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
summary = '\n'.join([
    f'# 18 프롬프트 + 거절 규칙 (v2.1){" [QUICK 점검]" if QUICK else ""} ({stamp} KST)', '',
    f'- 최종 모델: {FINAL} (생성 {GEN_NAME}{" + DPO" if ADAPTER else ""}, 검색 {RETR})',
    f'- **고른 프롬프트 {best_p} / 거절 규칙 {best_rule}** (검증 균형 점수 = (정답률 + 진짜 답 없음 거절률)/2)',
    f'- 시험 정답률 최종 - 08: {ci[0]:+.3f} (95% CI {ci[1]:+.3f} ~ {ci[2]:+.3f})', '',
    '## 검증', '', val_tab.to_markdown(), '', '## 시험 650', '', test_tab.to_markdown(), '',
    '## 검증 전체 그리드 (균형 점수 순)', '', grid_tab.sort_values('균형 점수', ascending=False).to_markdown(), '',
    '## final_config.json', '', '```json', json.dumps(final_cfg, ensure_ascii=False, indent=1), '```',
])
paths = [f'{OUT_DIR}/results_summary.md'] + ([] if QUICK else [f'{REPORT_DIR}/18_final_summary.md'])
for path in paths:
    with open(path, 'w', encoding='utf-8') as f:
        f.write(summary)
print(summary)

## 10. 드라이브 저장 마무리 + 런타임 종료 (QUICK 점검일 땐 런타임 유지)

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if not QUICK:
    runtime.unassign()